# Imports

In [1]:
import json
import pandas as pd
import os

# Prepare synthetic dataset

In [2]:
path = "/mnt/md0/synvoices/data/hausa_yourtts_asr_augmented/manifest.jsonl"

with open(path, "r") as f:
    lines = f.readlines()
    data = [json.loads(line) for line in lines]

len(data)

622084

In [3]:
data[0]

{'audio_filepath': '/mnt/md0/synvoices/data/hausa_yourtts_asr_augmented/clips/bec03149a8985b695e3272984df59562.wav',
 'text': 'akwai bukatar kare muhallin mu daga gurbatawa',
 'original_text': 'akwai bukatar kare muhallin mu daga gurbatawa.',
 'duration': 5.11}

In [4]:
df = pd.DataFrame(data)
df.describe(include="all")

,audio_filepath,text,original_text,duration
count,622084,622084,622084,622084.000000
unique,622084,621443,622084,NaN
top,/mnt/md0/synvoices/data/hausa_yourtts_asr_augm...,za mu iya yin wasa a bakin teku,shin ka taɓa ganin babur a kasuwa?,NaN
freq,1,2,1,NaN
mean,NaN,NaN,NaN,5.754159
std,NaN,NaN,NaN,1.065012
min,NaN,NaN,NaN,1.952687
25%,NaN,NaN,NaN,5.014000
50%,NaN,NaN,NaN,5.686000
75%,NaN,NaN,NaN,6.422000


In [5]:
# strip leading and trailing whitespace from the text column
df["text"] = df["text"].str.strip()

# drop duplicates
df = df.drop_duplicates(subset=["text"])
df.describe(include="all")

,audio_filepath,text,original_text,duration
count,621441,621441,621441,621441.000000
unique,621441,621441,621441,NaN
top,/mnt/md0/synvoices/data/hausa_yourtts_asr_augm...,shin ka taɓa ganin babur a kasuwa,shin ka taɓa ganin babur a kasuwa?,NaN
freq,1,1,1,NaN
mean,NaN,NaN,NaN,5.754452
std,NaN,NaN,NaN,1.065076
min,NaN,NaN,NaN,1.952687
25%,NaN,NaN,NaN,5.014000
50%,NaN,NaN,NaN,5.686000
75%,NaN,NaN,NaN,6.422000


In [6]:
# remove samples longer than 15 seconds
df = df[df["duration"] <= 15]
df["duration"].describe()

count    621398.000000
mean          5.753373
std           1.055607
min           1.952687
25%           5.014000
50%           5.686000
75%           6.422000
max          14.902000
Name: duration, dtype: float64

In [7]:
total_duration = df['duration'].sum()
# total duration in hours
total_duration / 3600

np.float64(993.092907065972)

In [8]:
cols = ['audio_filepath', 'text', 'duration']
df = df[cols]

# save to new manifest
new_manifest_path = os.path.join(
    os.path.dirname(path),
    f"manifest_{int(total_duration / 3600)}h.jsonl"
)
df.to_json(new_manifest_path, orient="records", lines=True)

In [9]:
def sample_dataset(df, duration, manifest_name, random_state=1):
    """
    Sample a dataset to a specific duration.
    """
    total_duration = float(df['duration'].sum())
    ratio = (duration * 3600) / total_duration
    n = int(len(df) * ratio)
    sample = df.sample(n=n, random_state=random_state)
    sample = sample.sample(frac=1, random_state=random_state).reset_index(drop=True)
    sample_duration = sample['duration'].sum()
    print(f"Sample duration: {sample_duration / 3600:.2f} hours")
    
    # save to new manifest
    sample_manifest_path = os.path.join(
        os.path.dirname(path),
        f"{manifest_name}_{int(duration)}h.jsonl"
    )
    sample.to_json(sample_manifest_path, orient="records", lines=True)
    print(f"Manifest saved to {sample_manifest_path}")

In [ ]:
sample_dataset(df, 250, "manifest", random_state=7)

Sample duration: 250.01 hours
Manifest saved to /mnt/md0/synvoices/data/hausa_yourtts_asr_augmented/manifest_250h.jsonl


In [25]:
sample_dataset(df, 400, "manifest", random_state=6)

Sample duration: 400.00 hours
Manifest saved to /mnt/md0/synvoices/data/hausa_yourtts_asr_augmented/manifest_400h.jsonl
